This file stores my solutions for 
DB PRO'S ADVENT OF SQL 2025

which can be found here: https://www.dbpro.app/advent-of-sql

In [ ]:
-- DAY 1
SELECT name
FROM reindeer
WHERE
    id NOT IN(
        SELECT id
        FROM reindeer
            INNER JOIN checkins ON reindeer.id = checkins.reindeer_id
            AND checkins.checkin_date = '2025-12-01'
    );

In [ ]:
-- DAY 2

SELECT name
FROM elves
    INNER JOIN (
        SELECT elf_id, MAX(win.c)
        FROM (
                SELECT checkin_date, elf_id, COUNT(*) AS c
                FROM checkins
                    INNER JOIN elf_checkins ON elf_checkins.work_date = checkins.checkin_date
                WHERE
                    reindeer_id = 8
                GROUP BY
                    elf_id
            ) win
    ) AS chat ON elves.id = chat.elf_id;

In [ ]:
-- DAY 3
SELECT location
FROM (
        SELECT location, MAX(num_of_visits_loc.c)
        FROM (
                SELECT location, visit_date, COUNT(*) AS c
                FROM locations_visited
                WHERE
                    reindeer_id = 8
                GROUP BY
                    location
            ) AS num_of_visits_loc
    ) AS answer_row;

In [ ]:
-- DAY 4
SELECT GROUP_CONCAT(char_1, '')
FROM (
        SELECT word, word_position, SUBSTRING(word, 1, 1) AS char_1
        FROM clearing_messages
            INNER JOIN reindeer on reindeer.id = clearing_messages.reindeer_id
        WHERE
            reindeer.name = = 'Blitzen'
        ORDER BY word_position ASC
    );

In [ ]:
-- DAY 5
SELECT COUNT(*) AS c
FROM (
        SELECT AVG(temp_celsius) AS AvgTemp, reading_date
        FROM temperature_readings
        GROUP BY
            reading_date
    )
WHERE
    AvgTemp > 0;

In [ ]:
-- DAY 6
SELECT
    name,
    expected_total,
    total_quantity
FROM
    toys
    INNER JOIN production_summary ON toys.id = production_summary.toy_id
    INNER JOIN (
        SELECT toy_id, SUM(quantity) AS total_quantity
        FROM production_logs
        GROUP BY
            production_logs.toy_id
    ) AS total_toy_quantity ON production_summary.toy_id = total_toy_quantity.toy_id
WHERE
    expected_total != total_quantity;

In [ ]:
-- DAY 7
SELECT sg, MAX(common_times)
FROM (
        SELECT REVERSE(sub_group) as sg, COUNT(*) as common_times
        FROM (
                SELECT error_code, SUBSTRING(
                        REVERSE(error_code), INSTR(REVERSE(error_code), '_') + 1
                    ) AS sub_group
                FROM machine_errors
            )
        GROUP BY
            sub_group
    )

In [ ]:
-- DAY 8
SELECT GROUP_CONCAT(letter, '')
FROM (
        SELECT position, rune_id, letter
        FROM teleport_sequence
            INNER JOIN runes ON teleport_sequence.rune_id = runes.id
        ORDER BY position
    )

In [ ]:
-- DAY 9
SELECT sig_hash, energy
FROM teleport_log
WHERE
    sig_hash NOT IN(
        SELECT sig_hash
        FROM known_beings
    )
ORDER BY energy DESC;

In [ ]:
-- DAY 10
SELECT creature, fragment, frag
FROM (
        creature_traits
        LEFT JOIN (
            SELECT sig_hash, fragment as frag
            FROM rune_fragments
            WHERE
                sig_hash = 'VOID-7F3C'
        ) as temp ON creature_traits.fragment = temp.frag
    ) as temp2
GROUP BY
    creature
HAVING
    COUNT(fragment) = COUNT(frag)

In [ ]:
-- DAY 11
SELECT
    weapon,
    property,
    prop,
    prop2,
    COUNT(property) AS num_proptery,
    COUNT(prop) AS num_prop,
    COUNT(prop2) AS num_prop2
FROM
    weapon_properties
    LEFT JOIN (
        SELECT property AS prop
        FROM creature_weaknesses
        WHERE
            type = 'weakness'
    ) as weakness ON weapon_properties.property = weakness.prop
    LEFT JOIN (
        SELECT property AS prop2
        FROM creature_weaknesses
        WHERE
            type = 'forbidden'
    ) AS forbidden ON weapon_properties.property = forbidden.prop2
GROUP BY
    weapon
HAVING
    COUNT(prop) = (
        SELECT COUNT(*)
        FROM creature_weaknesses
        WHERE
            type = 'weakness'
    )
    AND COUNT(prop2) = 0

In [ ]:
-- DAY 12
SELECT body_part, SUM(multiplier * base_damage) as damage
FROM body_part_affinities
    LEFT JOIN (
        SELECT property as prop, base_damage
        FROM property_effects
    ) as pe ON body_part_affinities.property = pe.prop
GROUP BY
    body_part --HAVING SUM(multiplier * base_damage)
ORDER BY damage DESC

In [ ]:
-- DAY 13
SELECT COUNT(*)
FROM (
        SELECT COUNT(*) AS num_kids
        FROM behaviour_events
        GROUP BY
            child_id
        HAVING
            COUNT(category) > 1
    ) AS naughty_childeren

In [ ]:
-- Day 14
SELECT recieve, COUNT(recieve) as num_case
FROM (
        SELECT
            CASE naughty_events
                WHEN 0 THEN 'Toy'
                WHEN 1 THEN 'Book'
                ELSE 'Coal'
            END recieve
        FROM (
                SELECT *, COUNT(child_id) as naughty_events
                FROM children
                    LEFT JOIN behaviour_events ON children.id = behaviour_events.child_id
                GROUP BY
                    name
            )
    )
GROUP BY
    recieve

In [ ]:
-- leetcode LOL SQL50
SELECT Department.name AS department, ranked_emp.name AS employee, salary
FROM (
        SELECT departmentId, name, salary, DENSE_RANK() OVER (
                PARTITION BY
                    departmentId
                ORDER BY salary DESC
            ) as rank
        FROM Employee
    ) as ranked_emp
    INNER JOIN Department ON ranked_emp.departmentId = Department.id
WHERE
    rank < 4

In [ ]:
-- DAY 15
SELECT present1, quantity, num_needed, (num_needed - IFNULL(quantity, 0) ) AS missing
FROM inventory RIGHT JOIN
(SELECT present AS present1, COUNT(present) AS num_needed 
FROM present_assignments  
GROUP BY present1) AS needed
ON needed.present1 = inventory.present 
ORDER BY missing DESC